# Notebook 17 — NeuralHydrology: Regional EA-LSTM

Implements the NeuralHydrology framework (Kratzert et al. 2022, JOSS) for regional EA-LSTM
training on DO prediction across all CAMELS-CH-Chem gauges with DO data.

Reference: https://github.com/neuralhydrology/neuralhydrology  
Motivation: Thiago Nascimento (Eawag) — regional EA-LSTM is state-of-the-art for
ungauged prediction; more gauges = better generalization.

Comparison target: manual EA-LSTM (nb04) achieves 0.420 mg/L RMSE.

In [ ]:
import subprocess, sys
# Install neuralhydrology if not present
# Install dependencies using the kernel's own Python executable
for pkg in ["neuralhydrology", "xarray", "ruamel.yaml", "numba", "tensorboard", "h5py", "protobuf"]:
    try:
        __import__(pkg)
        print(f"{pkg} already installed")
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"], check=True)
        print(f"{pkg} installed")

import neuralhydrology
print("All dependencies ready")

import os, sys, warnings, json
import xarray as xr
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

REPO_ROOT = Path('/storage/homefs/tn20y076/AareML')
sys.path.insert(0, str(REPO_ROOT))

from src.config import TRAIN_END, VAL_END, SEED, RESULTS_DIR, FIGURES_DIR

NH_DATA_DIR   = REPO_ROOT / 'data' / 'nh_dataset'   # NeuralHydrology formatted data
NH_RUN_DIR    = REPO_ROOT / 'results' / 'nh_run'      # NeuralHydrology output directory
NH_CONFIG     = REPO_ROOT / 'neuralhydrology_config.yml'

CHEM_DAILY = REPO_ROOT / 'data/camels-ch-chem/stream_water_chemistry/timeseries/daily'
BASE_ATTRS  = REPO_ROOT / 'data/camels-ch-base/camels_ch_attributes.csv'
LANDCOVER_DIR = REPO_ROOT / 'data/camels-ch-chem/catchment_aggregated_data/landcover_data'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print("NeuralHydrology: ready")

# DO gauges (16 gauges with sufficient DO coverage)
DO_GAUGES = [2473, 2009, 2613, 2143, 2016, 2174, 2415,
             2044, 2410, 2085, 2462, 2018, 2243, 2068, 2135, 2130]
print(f"DO gauges: {len(DO_GAUGES)}")

## 1. Data Preparation — Convert to NeuralHydrology Format

In [ ]:
# NeuralHydrology GenericDataset requires:
# data/
#   time_series/
#     <gauge_id>.nc   — one NetCDF per gauge with all time series
#   attributes/
#     attributes.csv  — static attributes indexed by gauge_id

NH_DATA_DIR.mkdir(parents=True, exist_ok=True)
ts_dir   = NH_DATA_DIR / 'time_series'
attr_dir = NH_DATA_DIR / 'attributes'
ts_dir.mkdir(exist_ok=True)
attr_dir.mkdir(exist_ok=True)

# Load base attributes
base_df = pd.read_csv(BASE_ATTRS, comment='#')
base_df['gauge_id'] = base_df['gauge_id'].astype(str)

# Load landcover (most recent year per gauge)
def load_landcover(gid):
    lc_file = LANDCOVER_DIR / f'camels_ch_chem_landcover_{gid}.csv'
    if not lc_file.exists():
        return {}
    df = pd.read_csv(lc_file)
    df['date'] = pd.to_datetime(df['date'])
    latest = df.sort_values('date').iloc[-1]
    return {
        'forest_frac': (latest.get('dwood_perc', 0) + latest.get('ewood_perc', 0) + latest.get('mixed_wood_perc', 0)) / 100,
        'crop_frac':   latest.get('crop_perc', 0) / 100,
        'urban_frac':  latest.get('urban_perc', 0) / 100,
        'ice_frac':    latest.get('ice_perc', 0) / 100,
    }

print("Converting time series to NetCDF...")
converted = []
for f in sorted(CHEM_DAILY.glob('camels_ch_chem_daily_*.csv')):
    gid = f.stem.split('_')[-1]
    try:
        df = pd.read_csv(f, parse_dates=['date'], index_col='date')
        # Rename columns to standard names
        df = df.rename(columns={
            'temp_sensor':  'water_temperature',
            'pH_sensor':    'pH',
            'ec_sensor':    'EC',
            'O2C_sensor':   'dissolved_oxygen',
        })
        # Create xarray dataset
        ds = xr.Dataset.from_dataframe(df)
        # Ensure time coordinate is encoded as numeric (days since epoch)
        # to avoid NeuralHydrology's string date parser using %d/%m/%Y
        encoding = {'date': {'units': 'days since 1970-01-01', 'dtype': 'float64'}}
        ds.to_netcdf(ts_dir / f'{gid}.nc', encoding=encoding)
        converted.append(gid)
    except Exception as e:
        print(f"  {gid}: {e}")

print(f"Converted {len(converted)} gauge time series to NetCDF")

# Build attributes CSV
print("Building static attributes...")
attr_rows = []
for gid in converted:
    base_row = base_df[base_df['gauge_id'].astype(str) == str(gid)]
    if base_row.empty:
        continue
    b = base_row.iloc[0]
    lc = load_landcover(gid)
    attr_rows.append({
        'gauge_id':    gid,
        'log_area':    np.log1p(float(b.get('area', 1))),
        'elev_mean':   float(b.get('elev_mean', 0)),
        'aridity':     float(b.get('aridity', 0)),
        'p_mean':      float(b.get('p_mean', 0)),
        'frac_snow':   float(b.get('frac_snow', 0)),
        'runoff_ratio':float(b.get('runoff_ratio', 0)),
        'pet_mean':    float(b.get('pet_mean', 0)),
        'forest_frac': lc.get('forest_frac', 0),
        'crop_frac':   lc.get('crop_frac', 0),
        'urban_frac':  lc.get('urban_frac', 0),
    })

attrs_df = pd.DataFrame(attr_rows).set_index('gauge_id')
attrs_df.to_csv(attr_dir / 'attributes.csv')
print(f"Attributes saved: {len(attrs_df)} gauges × {len(attrs_df.columns)} features")
print(attrs_df.head(3))

In [ ]:
# Write basin list files
basin_file_train = NH_DATA_DIR / 'basins_train.txt'
basin_file_val   = NH_DATA_DIR / 'basins_val.txt'
basin_file_test  = NH_DATA_DIR / 'basins_test.txt'

# Use all DO gauges for training; test on all
do_gauge_strs = [str(g) for g in DO_GAUGES if str(g) in converted]
with open(basin_file_train, 'w') as f: f.write('\n'.join(do_gauge_strs))
with open(basin_file_val,   'w') as f: f.write('\n'.join(do_gauge_strs))
with open(basin_file_test,  'w') as f: f.write('\n'.join(do_gauge_strs))
print(f"Basin files written: {len(do_gauge_strs)} DO gauges")

# Write NeuralHydrology YAML config
config_yaml = f"""
# AareML — NeuralHydrology EA-LSTM configuration
# Regional training on CAMELS-CH-Chem DO gauges
# Reference: Kratzert et al. 2022, JOSS

experiment_name: aareml_ealstm
run_dir: {NH_RUN_DIR}
train_basin_file: {basin_file_train}
validation_basin_file: {basin_file_val}
test_basin_file: {basin_file_test}

# Data
dataset: generic
data_dir: {NH_DATA_DIR}
dynamic_inputs:
  - water_temperature
  - pH
  - EC
target_variables:
  - dissolved_oxygen
static_attributes:
  - log_area
  - elev_mean
  - aridity
  - p_mean
  - frac_snow
  - runoff_ratio
  - pet_mean
  - forest_frac
  - crop_frac
  - urban_frac

# Time periods (match AareML train/val/test split)
train_start_date: \"01/01/1981\"
train_end_date: \"31/12/2014\"
validation_start_date: \"01/01/2015\"
validation_end_date: \"31/12/2016\"
test_start_date: \"01/01/2017\"
test_end_date: \"31/12/2020\"

# Sequence
seq_length: 21
predict_last_n: 14

# Model
model: ealstm
head: regression
hidden_size: 128
initial_forget_bias: 3
output_dropout: 0.4
batch_size: 256
epochs: 30
learning_rate:
  0: 0.001
  10: 0.0005
optimizer: Adam
clip_gradient_norm: 1.0
loss: NSE

# Metrics
metrics:
  - NSE
  - RMSE
  - KGE

# Other
log_tensorboard: False
save_git_diff: False
seed: {SEED}
device: {DEVICE}
num_workers: 4
"""

with open(NH_CONFIG, 'w') as f:
    f.write(config_yaml)
print(f"Config written: {NH_CONFIG}")

## 2. Train EA-LSTM with NeuralHydrology

In [ ]:
from neuralhydrology.nh_run import start_run
from neuralhydrology.utils.config import Config
# Patch get_git_hash to avoid CalledProcessError on non-git install dirs
import neuralhydrology.utils.logging_utils as _lu
_lu.get_git_hash = lambda: None

NH_RUN_DIR.mkdir(parents=True, exist_ok=True)
cfg = Config(NH_CONFIG)

print("Starting NeuralHydrology EA-LSTM training...")
print(f"  Gauges: {len(do_gauge_strs)}")
print(f"  Target: dissolved_oxygen")
print(f"  Static attributes: {len(attrs_df.columns)}")
print(f"  Hidden size: 128, Epochs: 30")

start_run(config_file=NH_CONFIG, gpu=0 if DEVICE=='cuda' else -1)
print("Training complete.")

## 3. Evaluate and Compare to Manual EA-LSTM

In [ ]:
from neuralhydrology.nh_run import eval_run
from neuralhydrology.evaluation import get_tester

# Find the run directory (NeuralHydrology creates timestamped subdirectory)
run_dirs = sorted(NH_RUN_DIR.glob('aareml_ealstm_*'))
if not run_dirs:
    print("No run directory found — training may have failed")
else:
    latest_run = run_dirs[-1]
    print(f"Evaluating run: {latest_run.name}")
    
    eval_run(run_dir=latest_run, period='test', gpu=0 if DEVICE=='cuda' else -1)
    
    # Load results
    results_file = latest_run / 'test' / 'model_epoch030' / 'test_results.p'
    if results_file.exists():
        import pickle
        with open(results_file, 'rb') as f:
            results = pickle.load(f)
        
        # Extract per-gauge RMSE
        nh_rmse = []
        for gid, res in results.items():
            if 'dissolved_oxygen' in res:
                rmse = res['dissolved_oxygen']['RMSE']
                nse  = res['dissolved_oxygen']['NSE']
                nh_rmse.append({'gauge_id': gid, 'rmse_do': rmse, 'nse_do': nse, 'model': 'NH_EA-LSTM'})
        
        nh_df = pd.DataFrame(nh_rmse)
        nh_df = nh_df.dropna(subset=['RMSE', 'NSE', 'KGE'])
        print(f"\n=== NeuralHydrology EA-LSTM Results ===")
        print(f"Gauges evaluated: {len(nh_df)}")
        print(f"Mean RMSE: {nh_df.RMSE.mean():.4f} mg/L")
        print(f"Mean NSE:  {nh_df.NSE.mean():.3f}")
        
        print(f"\n=== Comparison ===")
        print(f"Manual EA-LSTM (nb04): 0.420 mg/L RMSE, NSE=0.843")
        print(f"NH EA-LSTM (nb17):     {nh_df.RMSE.mean():.3f} mg/L RMSE, NSE={nh_df.NSE.mean():.3f}")
        
        # Save
        nh_df.to_csv(RESULTS_DIR / 'nh_ealstm_results.csv', index=False)
        print(f"\nSaved: nh_ealstm_results.csv")

In [ ]:
# Plot comparison: Manual EA-LSTM vs NH EA-LSTM per gauge
if 'nh_df' in dir() and not nh_df.empty:
    manual_ea = pd.read_csv(RESULTS_DIR / 'ea_lstm_results.csv')
    
    fig, ax = plt.subplots(figsize=(10, 5))
    
    x = np.arange(len(nh_df))
    w = 0.35
    ax.bar(x - w/2, manual_ea.set_index('gauge_id').loc[nh_df.gauge_id, 'rmse_do'].values,
           w, label='Manual EA-LSTM (nb04)', color='#01696F', alpha=0.8)
    ax.bar(x + w/2, nh_df.rmse_do.values,
           w, label='NeuralHydrology EA-LSTM (nb17)', color='#0C4E54', alpha=0.8)
    
    ax.axhline(0.420, color='#01696F', ls='--', lw=1.5, alpha=0.5)
    ax.axhline(nh_df.rmse_do.mean(), color='#0C4E54', ls='--', lw=1.5, alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels(nh_df.gauge_id, rotation=45)
    ax.set_ylabel('DO RMSE (mg/L)'); ax.set_title('Manual EA-LSTM vs NeuralHydrology EA-LSTM')
    ax.legend(); plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'nb17_nh_comparison.png', dpi=150)
    plt.close()
    print("Figure saved: nb17_nh_comparison.png")